In [ ]:
# Cell 1: Install all required packages
!pip install sentence-transformers faiss-cpu streamlit pyngrok flask pandas numpy -q

print("✅ All packages installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 85.9 MB/s eta 0:00:00
✅ All packages installed successfully!


In [ ]:
!pip install pyvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.2 MB/s eta 0:00:00


In [ ]:
!pip install faiss-cpu sentence-transformers

In [ ]:
# Cell 2 (FIXED): Robust CSV loading with file validation
import pandas as pd
import os
import shutil
import zipfile

csv_path = '/content/Message Group - Product.csv'

print(f"📂 Checking file: {csv_path}")

# Check if file exists
if not os.path.exists(csv_path):
    print("❌ File not found!")
    print("\n💡 Please upload your CSV file:")
    print("1. Click the folder icon 📁 on the left")
    print("2. Click upload button")
    print("3. Select 'Message Group - Product.csv'")
    raise FileNotFoundError(f"File not found: {csv_path}")

# Check file size
file_size = os.path.getsize(csv_path)
print(f"📊 File size: {file_size / 1024:.2f} KB")

# Check if it's a ZIP file (PK header indicates ZIP)
with open(csv_path, 'rb') as f:
    header = f.read(2)

if header == b'PK':
    print("⚠️  This is a ZIP file, not a CSV!")
    print("🔄 Attempting to extract...")

    try:
        # Extract ZIP file
        with zipfile.ZipFile(csv_path, 'r') as zip_ref:
            zip_ref.extractall('/content/')
            extracted_files = zip_ref.namelist()
            print(f"✅ Extracted files: {extracted_files}")

            # Find the CSV file
            csv_file = [f for f in extracted_files if f.endswith('.csv')][0]
            csv_path = f'/content/{csv_file}'
            print(f"✅ Using extracted CSV: {csv_path}")
    except Exception as e:
        print(f"❌ Error extracting ZIP: {e}")
        raise

# Now read the CSV with multiple strategies
print("\n🔄 Loading CSV file...")

try:
    # Strategy 1: Standard read with latin-1
    df = pd.read_csv(csv_path, encoding='latin-1', low_memory=False)
    print("✅ Loaded with latin-1 encoding")
except Exception as e:
    print(f"⚠️  Strategy 1 failed: {e}")
    try:
        # Strategy 2: UTF-8 encoding
        df = pd.read_csv(csv_path, encoding='utf-8', low_memory=False, on_bad_lines='skip')
        print("✅ Loaded with utf-8 encoding")
    except Exception as e2:
        print(f"⚠️  Strategy 2 failed: {e2}")
        try:
            # Strategy 3: Line-by-line reading
            print("🔄 Trying line-by-line reading...")
            import csv

            rows = []
            with open(csv_path, 'r', encoding='latin-1', errors='ignore') as f:
                csv_reader = csv.reader(f)
                headers = next(csv_reader)

                for row in csv_reader:
                    if len(row) == len(headers):
                        rows.append(row)

            df = pd.DataFrame(rows, columns=headers)
            print("✅ Loaded with line-by-line parsing")
        except Exception as e3:
            print(f"❌ All strategies failed: {e3}")
            raise

# Validate the data
print(f"\n📊 Dataset loaded successfully!")
print(f"   ✅ Rows: {len(df)}")
print(f"   ✅ Columns: {len(df.columns)}")

print("\n📋 Column names:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i}. {col}")

# Show sample data
print("\n📝 First 3 rows:")
print(df.head(3))

# Save cleaned version
df.to_csv('products_clean.csv', index=False, encoding='utf-8')
print("\n✅ Saved clean version: products_clean.csv")

print("\n" + "="*60)
print("✅ CSV LOADED SUCCESSFULLY! Run Cell 3 next!")
print("="*60)

📂 Checking file: /content/Message Group - Product.csv
📊 File size: 730.75 KB

🔄 Loading CSV file...
✅ Loaded with latin-1 encoding

📊 Dataset loaded successfully!
   ✅ Rows: 4566
   ✅ Columns: 11

📋 Column names:
   1. S.No
   2. BrandName
   3. Product ID
   4. Product Name
   5. Brand Desc
   6. Product Size
   7. Currancy
   8. MRP
   9. SellPrice
   10. Discount
   11. Category

📝 First 3 rows:
   S.No BrandName Product ID              Product Name  \
0     1      4711      FR001         Cologne Fragrance   
1     2      109f       DRW1  DRW1 - Westernwear-Women   
2     3      109f       DRW2  DRW2 - Westernwear-Women   

                            Brand Desc                        Product Size  \
0            ekw eau de cologne 400 ml                               Small   
1  womens v- neck short dress - yellow  Size:Medium,Small,X-Large,XX-Large   
2  womens round neck solid top - black     Size:Large,Medium,Small,X-Large   

  Currancy   MRP  SellPrice Discount           Categ

In [ ]:
# Cell 3 (UPDATED): Build indexes using cleaned CSV
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import time

print("🔄 Loading cleaned CSV...")

# Use the cleaned CSV file
csv_path = 'products_clean.csv'
df = pd.read_csv(csv_path, encoding='utf-8')

print(f"📊 Total products loaded: {len(df)}")
print(f"📋 Columns: {list(df.columns)}")

# Drop rows with missing critical data
critical_columns = ['Product Name', 'Brand Desc', 'Category']
before_count = len(df)
df = df.dropna(subset=critical_columns, how='all')
after_count = len(df)

if before_count != after_count:
    print(f"🗑️  Dropped {before_count - after_count} rows with missing data")
print(f"✅ Valid products: {after_count}")

# Create working dataframe (drop metadata)
df_work = df.drop(columns=['S.No', 'BrandName', 'Product ID'], errors='ignore')
print(f"📋 Working columns: {list(df_work.columns)}")

# Create search text
def create_search_text(row):
    text_parts = []

    if pd.notna(row.get('Product Name')):
        text_parts.append(str(row['Product Name']).strip())

    if pd.notna(row.get('Brand Desc')):
        text_parts.append(str(row['Brand Desc']).strip())

    if pd.notna(row.get('Category')):
        text_parts.append(str(row['Category']).strip())

    if pd.notna(row.get('Product Size')):
        size = str(row['Product Size']).strip()
        if size and size.lower() not in ['nan', 'n/a']:
            text_parts.append(f"Size: {size}")

    if pd.notna(row.get('SellPrice')):
        price = str(row['SellPrice']).strip()
        if price and price not in ['nan', '0', '']:
            text_parts.append(f"Price: {price}")

    return " ".join(text_parts) if text_parts else "Product"

print("\n🔤 Creating search text...")
df_work['search_text'] = df_work.apply(create_search_text, axis=1)

# Filter valid products
df_work = df_work[df_work['search_text'].str.len() > 3]
df = df.loc[df_work.index]

print(f"✅ Products with valid search text: {len(df_work)}")
print("\n📝 Sample search texts:")
for i in range(min(3, len(df_work))):
    print(f"   {i+1}. {df_work['search_text'].iloc[i][:100]}...")

# Load embedding model
print("\n🤖 Loading Sentence Transformer model (all-MiniLM-L6-v2)...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model loaded!")

# Generate embeddings
print(f"\n🔢 Generating embeddings for {len(df_work)} products...")
embeddings = model.encode(
    df_work['search_text'].tolist(),
    show_progress_bar=True,
    batch_size=32,
    convert_to_numpy=True
)

print(f"✅ Embeddings generated!")
print(f"   Shape: {embeddings.shape}")
print(f"   Products: {embeddings.shape[0]}")
print(f"   Dimensions: {embeddings.shape[1]}")

# Prepare for FAISS
embeddings = embeddings.astype('float32')
faiss.normalize_L2(embeddings)
print("✅ Embeddings normalized")

dimension = embeddings.shape[1]
n_products = embeddings.shape[0]

print("\n" + "="*70)
print("🏗️  BUILDING 5 FAISS INDEXES")
print("="*70)

indexes = {}

# 1️⃣ Flat L2 (Euclidean Distance)
print("\n1️⃣ IndexFlatL2 (Euclidean Distance - EXACT)")
start = time.time()
index_flat_l2 = faiss.IndexFlatL2(dimension)
index_flat_l2.add(embeddings)
time_flat_l2 = time.time() - start
indexes['Flat L2 (Euclidean)'] = {
    'index': index_flat_l2,
    'build_time': time_flat_l2,
    'type': 'Exact Search',
    'memory': 'High',
    'precision': '100%'
}
print(f"   ✅ {time_flat_l2:.3f}s | {index_flat_l2.ntotal} products")

# 2️⃣ IVF Flat
print("\n2️⃣ IVFFlat (Inverted File Index)")
start = time.time()
nlist = max(4, min(int(np.sqrt(n_products)), 100))
quantizer = faiss.IndexFlatL2(dimension)
index_ivf = faiss.IndexIVFFlat(quantizer, dimension, nlist)
index_ivf.train(embeddings)
index_ivf.add(embeddings)
index_ivf.nprobe = min(10, max(1, nlist // 4))
time_ivf = time.time() - start
indexes['IVF Flat'] = {
    'index': index_ivf,
    'build_time': time_ivf,
    'type': 'Approximate',
    'memory': 'Medium',
    'precision': '95-99%'
}
print(f"   ✅ {time_ivf:.3f}s | {nlist} clusters | nprobe={index_ivf.nprobe}")

# 3️⃣ IVF PQ
print("\n3️⃣ IVFPQ (IVF + Product Quantization)")
start = time.time()
nlist_pq = max(4, min(int(np.sqrt(n_products)), 100))
m = 8
nbits = 8
quantizer_pq = faiss.IndexFlatL2(dimension)
index_ivfpq = faiss.IndexIVFPQ(quantizer_pq, dimension, nlist_pq, m, nbits)
index_ivfpq.train(embeddings)
index_ivfpq.add(embeddings)
index_ivfpq.nprobe = min(10, max(1, nlist_pq // 4))
time_ivfpq = time.time() - start
indexes['IVF PQ'] = {
    'index': index_ivfpq,
    'build_time': time_ivfpq,
    'type': 'Compressed Approximate',
    'memory': 'Low',
    'precision': '85-95%'
}
print(f"   ✅ {time_ivfpq:.3f}s | {m}x{nbits} compression")

# 4️⃣ HNSW
print("\n4️⃣ HNSW (Graph-based)")
start = time.time()
M = 32
index_hnsw = faiss.IndexHNSWFlat(dimension, M)
index_hnsw.hnsw.efConstruction = 40
index_hnsw.add(embeddings)
index_hnsw.hnsw.efSearch = 16
time_hnsw = time.time() - start
indexes['HNSW'] = {
    'index': index_hnsw,
    'build_time': time_hnsw,
    'type': 'Graph-based',
    'memory': 'High',
    'precision': '95-99%'
}
print(f"   ✅ {time_hnsw:.3f}s | M={M}")

# 5️⃣ PQ
print("\n5️⃣ PQ (Product Quantization)")
start = time.time()
m_pq = 8
nbits_pq = 8
index_pq = faiss.IndexPQ(dimension, m_pq, nbits_pq)
index_pq.train(embeddings)
index_pq.add(embeddings)
time_pq = time.time() - start
indexes['PQ Only'] = {
    'index': index_pq,
    'build_time': time_pq,
    'type': 'Compressed',
    'memory': 'Very Low',
    'precision': '80-90%'
}
print(f"   ✅ {time_pq:.3f}s | {m_pq}x{nbits_pq}")

print("\n" + "="*70)
print("🎉 ALL 5 INDEXES BUILT!")
print("="*70)

# Save everything
print("\n💾 Saving all data...")
with open('all_indexes.pkl', 'wb') as f:
    pickle.dump({
        'indexes': indexes,
        'df': df,
        'model': model,
        'embeddings': embeddings
    }, f)

print("✅ Saved to: all_indexes.pkl")

# Summary
print("\n📊 BUILD TIME SUMMARY:")
print("-" * 70)
for name, info in indexes.items():
    print(f"{name:25} {info['build_time']:>6.3f}s  ({info['type']}, {info['memory']} memory)")
print("-" * 70)

print("\n✅ Ready for Cell 4!")

🔄 Loading cleaned CSV...
📊 Total products loaded: 4566
📋 Columns: ['S.No', 'BrandName', 'Product ID', 'Product Name', 'Brand Desc', 'Product Size', 'Currancy', 'MRP', 'SellPrice', 'Discount', 'Category']
✅ Valid products: 4566
📋 Working columns: ['Product Name', 'Brand Desc', 'Product Size', 'Currancy', 'MRP', 'SellPrice', 'Discount', 'Category']

🔤 Creating search text...
✅ Products with valid search text: 4566

📝 Sample search texts:
   1. Cologne Fragrance ekw eau de cologne 400 ml Fragrance-Women Size: Small Price: 3120...
   2. DRW1 - Westernwear-Women womens v- neck short dress - yellow Westernwear-Women Size: Size:Medium,Sma...
   3. DRW2 - Westernwear-Women womens round neck solid top - black Westernwear-Women Size: Size:Large,Medi...

🤖 Loading Sentence Transformer model (all-MiniLM-L6-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded!

🔢 Generating embeddings for 4566 products...


Batches:   0%|          | 0/143 [00:00<?, ?it/s]

✅ Embeddings generated!
   Shape: (4566, 384)
   Products: 4566
   Dimensions: 384
✅ Embeddings normalized

🏗️  BUILDING 5 FAISS INDEXES

1️⃣ IndexFlatL2 (Euclidean Distance - EXACT)
   ✅ 0.002s | 4566 products

2️⃣ IVFFlat (Inverted File Index)
   ✅ 0.060s | 67 clusters | nprobe=10

3️⃣ IVFPQ (IVF + Product Quantization)
   ✅ 0.717s | 8x8 compression

4️⃣ HNSW (Graph-based)
   ✅ 0.255s | M=32

5️⃣ PQ (Product Quantization)
   ✅ 0.638s | 8x8

🎉 ALL 5 INDEXES BUILT!

💾 Saving all data...
✅ Saved to: all_indexes.pkl

📊 BUILD TIME SUMMARY:
----------------------------------------------------------------------
Flat L2 (Euclidean)        0.002s  (Exact Search, High memory)
IVF Flat                   0.060s  (Approximate, Medium memory)
IVF PQ                     0.717s  (Compressed Approximate, Low memory)
HNSW                       0.255s  (Graph-based, High memory)
PQ Only                    0.638s  (Compressed, Very Low memory)
------------------------------------------------------------

In [ ]:
# Cell 4 (FINAL VERSION – WITH SIDEBAR, PERFECT GRAPH, DETAILED RESULTS)

streamlit_code = '''
import streamlit as st
import pandas as pd
import pickle
import numpy as np
import time
import faiss
from pyvis.network import Network
import tempfile

# -------------------------------------------------
# Page config
# -------------------------------------------------
st.set_page_config(
    page_title="🔬 FAISS Index Comparison",
    page_icon="🔬",
    layout="wide",
    initial_sidebar_state="expanded"
)

# -------------------------------------------------
# Load saved data
# -------------------------------------------------
@st.cache_resource
def load_data():
    with open("all_indexes.pkl", "rb") as f:
        data = pickle.load(f)
    return data["indexes"], data["df"], data["model"]

indexes, df, model = load_data()

# -------------------------------------------------
# SIDEBAR
# -------------------------------------------------
with st.sidebar:
    st.header("📊 Database Info")
    st.metric("Total Products", f"{len(df):,}")

    if "Category" in df.columns:
        st.metric("Categories", df["Category"].nunique())

    st.markdown("---")
    st.subheader("🎯 Search Methods")

    st.markdown("""
**Flat L2 (Euclidean)**
- ✅ Most accurate (100%)
- ⚠️ Slower for big data

**IVF Flat**
- ✅ Good balance
- ✅ Groups similar items

**HNSW** 🌟
- ✅ Super fast ⚡
- ✅ Graph structure
- ✅ Visual neighbor graph

**PQ Only**
- ✅ Saves memory (48x)
- ⚠️ Slightly less accurate
""")

    st.markdown("---")
    st.subheader("💡 Tips")
    st.markdown("""
- Results show instantly
- Compare methods side-by-side
- HNSW graph is auto-fitted
""")

# -------------------------------------------------
# Title
# -------------------------------------------------
st.title("🔬 FAISS-Powered Semantic Product Search")
st.markdown("### Compare different FAISS search methods side-by-side")
st.markdown("---")

# -------------------------------------------------
# Search suggestions
# -------------------------------------------------
st.markdown("## 💡 Click to Try These Searches")

search_categories = {
    "🛍️ By Product Type": [
        "black dress for women",
        "casual summer top",
        "leather handbag",
        "running shoes",
        "perfume for women"
    ],
    "💰 By Price": [
        "affordable perfume under 500",
        "cheap lipstick",
        "budget friendly dress"
    ],
    "👗 By Style": [
        "formal shirt for office",
        "party wear dress",
        "casual weekend outfit"
    ],
    "🎨 By Color & Size": [
        "red top size medium",
        "black shoes size 8",
        "white dress small"
    ]
}

cols = st.columns(2)
for i, (cat, ideas) in enumerate(search_categories.items()):
    with cols[i % 2]:
        with st.expander(cat, expanded=(i < 2)):
            for idea in ideas:
                if st.button(f"🔍 {idea}", key=f"idea_{idea}", use_container_width=True):
                    st.session_state.search_query = idea
                    st.rerun()

st.markdown("---")

# -------------------------------------------------
# Search input
# -------------------------------------------------
if "search_query" not in st.session_state:
    st.session_state.search_query = ""

col1, col2 = st.columns([3, 1])
with col1:
    query = st.text_input(
        "What are you looking for?",
        value=st.session_state.search_query,
        placeholder="e.g., black dress, shoes, perfume"
    )

with col2:
    num_results = st.selectbox("Show results:", [5, 10, 15, 20], index=1)

# -------------------------------------------------
# Index selection
# -------------------------------------------------
st.markdown("### ⚙️ Choose Search Methods")

available_indexes = {k: v for k, v in indexes.items() if k != "IVF PQ"}
index_cols = st.columns(4)
selected_indexes = []

for i, (name, info) in enumerate(available_indexes.items()):
    with index_cols[i]:
        if st.checkbox(
            name,
            value=(name in ["Flat L2 (Euclidean)", "HNSW"]),
            key=f"chk_{name}"
        ):
            selected_indexes.append(name)
        st.caption(f"⏱️ {info['build_time']:.3f}s build")

st.markdown("---")

# -------------------------------------------------
# SEARCH
# -------------------------------------------------
if st.button("🚀 Search Now", type="primary", use_container_width=True):

    if not query:
        st.warning("⚠️ Please enter a search query.")
        st.stop()

    if not selected_indexes:
        st.warning("⚠️ Please select at least one search method.")
        st.stop()

    with st.spinner("🔄 Searching products..."):
        query_embedding = model.encode([query]).astype("float32")
        faiss.normalize_L2(query_embedding)

    all_results = {}

    for index_name in selected_indexes:
        start = time.time()
        index_obj = indexes[index_name]["index"]
        distances, indices = index_obj.search(query_embedding, num_results)
        elapsed = time.time() - start

        results = df.iloc[indices[0]].copy()
        results["distance"] = distances[0]
        results["similarity"] = 100 - np.clip(distances[0] * 10, 0, 100)

        all_results[index_name] = {
            "results": results,
            "time": elapsed,
            "indices": indices[0],
            "distances": distances[0]
        }

    st.success("✅ Search completed successfully!")
    st.markdown("---")

    # -------------------------------------------------
    # Speed comparison
    # -------------------------------------------------
    st.markdown("### ⚡ Speed Comparison")
    speed_cols = st.columns(len(all_results))
    for i, (name, data) in enumerate(all_results.items()):
        with speed_cols[i]:
            st.metric(name, f"{data['time']*1000:.2f} ms")

    st.markdown("---")

    # -------------------------------------------------
    # HNSW Graph – PERFECTLY FITTED
    # -------------------------------------------------
    if "HNSW" in all_results:
        with st.expander("📊 HNSW Neighborhood Graph", expanded=True):
            net = Network(
                height="900px",
                width="100%",
                bgcolor="#ffffff",
                directed=False
            )

            net.set_options("""
            var options = {
              "interaction": {
                "zoomView": false,
                "dragView": true
              },
              "nodes": {
                "shape": "dot",
                "size": 26,
                "font": { "size": 18 }
              },
              "edges": {
                "smooth": true
              },
              "physics": {
                "enabled": true,
                "barnesHut": {
                  "gravitationalConstant": -1200,
                  "centralGravity": 0.25,
                  "springLength": 150,
                  "springConstant": 0.05,
                  "avoidOverlap": 0.7
                },
                "stabilization": {
                  "iterations": 200
                }
              }
            }
            """)

            ids = all_results["HNSW"]["indices"][:12]
            dists = all_results["HNSW"]["distances"][:12]

            for i, idx in enumerate(ids):
                name = df.iloc[idx]["Product Name"][:40]
                sim = 100 - np.clip(dists[i] * 10, 0, 100)
                net.add_node(
                    i,
                    label=str(i + 1),
                    title=f"{name}\\nMatch: {sim:.1f}%"
                )

            for i in range(len(ids)):
                for j in range(i + 1, min(i + 4, len(ids))):
                    net.add_edge(i, j)

            tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".html")
            net.save_graph(tmp.name)
            st.components.v1.html(open(tmp.name).read(), height=930)

    # -------------------------------------------------
    # Search Results (DETAILED)
    # -------------------------------------------------
    st.markdown("### 🎯 Search Results")

    result_cols = st.columns(len(all_results))
    for col, (name, data) in zip(result_cols, all_results.items()):
        with col:
            st.markdown(f"## {name}")
            st.markdown(f"⏱️ {data['time']*1000:.2f} ms")
            st.markdown("---")

            for _, row in data["results"].iterrows():
                st.markdown(f"### 🛍️ {row.get('Product Name','N/A')}")

                if pd.notna(row.get("Brand Desc")):
                    st.markdown(f"**Brand:** {row['Brand Desc']}")
                if pd.notna(row.get("Category")):
                    st.markdown(f"**Category:** {row['Category']}")

                c1, c2 = st.columns(2)
                with c1:
                    if pd.notna(row.get("MRP")) and row["MRP"] != 0:
                        st.markdown(f"~~₹ {row['MRP']}~~")
                with c2:
                    if pd.notna(row.get("SellPrice")) and row["SellPrice"] != 0:
                        st.markdown(f"**💰 ₹ {row['SellPrice']}**")

                if pd.notna(row.get("Discount")):
                    st.markdown(f"🏷️ **Discount:** {row['Discount']}")
                if pd.notna(row.get("Product Size")):
                    st.markdown(f"📏 **Size:** {row['Product Size']}")

                st.progress(row["similarity"] / 100)
                st.markdown(f"🎯 **Match:** {row['similarity']:.1f}%")
                st.markdown("---")
'''

with open("app_comparison.py", "w") as f:
    f.write(streamlit_code)

print("✅ CELL 4 FINAL VERSION READY")
print("✅ Graph auto-fits screen")
print("✅ No zoom hassle")
print("✅ Detailed results + sidebar intact")


✅ CELL 4 FINAL VERSION READY
✅ Graph auto-fits screen
✅ No zoom hassle
✅ Detailed results + sidebar intact


In [ ]:
# Cell 5: Setup ngrok authentication
from pyngrok import ngrok
import getpass

# Get ngrok auth token
print("🔑 Get your FREE ngrok auth token from:")
print("   👉 https://dashboard.ngrok.com/get-started/your-authtoken")
print("\n")
auth_token = getpass.getpass("Enter your ngrok auth token: ")

# Set ngrok auth token
ngrok.set_auth_token(auth_token)

print("✅ ngrok authentication successful!")

🔑 Get your FREE ngrok auth token from:
   👉 https://dashboard.ngrok.com/get-started/your-authtoken


Enter your ngrok auth token: ··········
✅ ngrok authentication successful!


In [ ]:
# Cell 6: Run comparison Streamlit app
import subprocess
import time
from pyngrok import ngrok
import os

# Kill any existing streamlit processes
print("🔄 Cleaning up old processes...")
!pkill -f streamlit
time.sleep(2)

# Start ngrok tunnel
print("🌐 Starting ngrok tunnel...")
public_url = ngrok.connect(8501)

print("\n" + "="*60)
print("🎉 YOUR COMPARISON APP IS LIVE!")
print("="*60)
print(f"🔗 URL: {public_url}")
print("="*60)

# Start Streamlit in background
print("\n🚀 Starting Streamlit comparison app...")

with open('start_comparison.sh', 'w') as f:
    f.write('#!/bin/bash\n')
    f.write('streamlit run app_comparison.py --server.port=8501 --server.headless=true > /dev/null 2>&1 &\n')

!chmod +x start_comparison.sh
!./start_comparison.sh

print("⏳ Waiting 10 seconds for app to start...")
time.sleep(10)

result = !ps aux | grep streamlit | grep -v grep
if result:
    print("✅ Comparison app is running!")
    print(f"🎉 Open your app: {public_url}")
    print("\n" + "="*60)
    print("✅ NOW COMPARE ALL INDEXES SIDE-BY-SIDE!")
    print("="*60)
else:
    print("⚠️ Streamlit may not have started. Check logs.")

print("\n✅ Ready! Run Cell 7 for performance testing!")

🔄 Cleaning up old processes...
🌐 Starting ngrok tunnel...

🎉 YOUR COMPARISON APP IS LIVE!
🔗 URL: NgrokTunnel: "https://charline-pearlier-khloe.ngrok-free.dev" -> "http://localhost:8501"

🚀 Starting Streamlit comparison app...
⏳ Waiting 10 seconds for app to start...
✅ Comparison app is running!
🎉 Open your app: NgrokTunnel: "https://charline-pearlier-khloe.ngrok-free.dev" -> "http://localhost:8501"

✅ NOW COMPARE ALL INDEXES SIDE-BY-SIDE!

✅ Ready! Run Cell 7 for performance testing!


In [ ]:
# Cell 7: Benchmark all indexes
import pandas as pd
import pickle
import numpy as np
import time
from sentence_transformers import SentenceTransformer

print("📊 Loading data for benchmarking...")
with open('all_indexes.pkl', 'rb') as f:
    data = pickle.load(f)

indexes = data['indexes']
model = data['model']
df = data['df']

# Test queries
test_queries = [
    "black dress for women",
    "affordable perfume",
    "casual top size medium",
    "evening wear under 1000",
    "men's cologne",
    "red lipstick",
    "denim jeans",
    "running shoes",
    "leather bag",
    "winter coat"
]

print(f"\n🔬 Running benchmark with {len(test_queries)} queries...")
print(f"📈 Testing {len(indexes)} indexes\n")

results_table = []

for index_name, index_data in indexes.items():
    print(f"Testing {index_name}...")
    index_obj = index_data['index']

    search_times = []

    for query in test_queries:
        query_emb = model.encode([query]).astype('float32')

        start = time.time()
        distances, indices = index_obj.search(query_emb, 10)
        elapsed = time.time() - start

        search_times.append(elapsed * 1000)  # Convert to ms

    avg_time = np.mean(search_times)
    std_time = np.std(search_times)
    min_time = np.min(search_times)
    max_time = np.max(search_times)

    results_table.append({
        'Index': index_name,
        'Type': index_data['type'],
        'Build Time (s)': f"{index_data['build_time']:.3f}",
        'Avg Search (ms)': f"{avg_time:.2f}",
        'Std Dev (ms)': f"{std_time:.2f}",
        'Min (ms)': f"{min_time:.2f}",
        'Max (ms)': f"{max_time:.2f}"
    })

# Create results DataFrame
results_df = pd.DataFrame(results_table)

print("\n" + "="*80)
print("📊 PERFORMANCE COMPARISON RESULTS")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

# Save results
results_df.to_csv('benchmark_results.csv', index=False)
print("\n✅ Results saved to: benchmark_results.csv")

# Recommendations
print("\n💡 RECOMMENDATIONS:")
print("="*80)
print("🏆 Flat L2: Best PRECISION (exact search), slower for large datasets")
print("⚡ IVF Flat: Good balance of SPEED and PRECISION")
print("💾 IVF PQ: Best for MEMORY efficiency, slight precision loss")
print("🚀 HNSW: Fastest SEARCH, higher memory usage")
print("📦 PQ Only: Most COMPRESSED, lower precision")
print("="*80)

📊 Loading data for benchmarking...

🔬 Running benchmark with 10 queries...
📈 Testing 5 indexes

Testing Flat L2 (Euclidean)...
Testing IVF Flat...
Testing IVF PQ...
Testing HNSW...
Testing PQ Only...

📊 PERFORMANCE COMPARISON RESULTS
              Index                   Type Build Time (s) Avg Search (ms) Std Dev (ms) Min (ms) Max (ms)
Flat L2 (Euclidean)           Exact Search          0.003            0.84         0.08     0.75     0.98
           IVF Flat            Approximate          0.143            0.25         0.04     0.20     0.34
             IVF PQ Compressed Approximate          1.478            0.19         0.03     0.16     0.27
               HNSW            Graph-based          0.420            0.16         0.06     0.11     0.33
            PQ Only             Compressed          0.859            0.24         0.05     0.19     0.33

✅ Results saved to: benchmark_results.csv

💡 RECOMMENDATIONS:
🏆 Flat L2: Best PRECISION (exact search), slower for large datasets
⚡ IVF

**This is for just used by normal serach not bashbord**

In [ ]:
# Debug Cell: Check the saved data structure
import pickle
import pandas as pd

# Load the saved data
with open('faiss_index.pkl', 'rb') as f:
    data = pickle.load(f)
    df_saved = data['df']

print("📊 Checking saved DataFrame...")
print(f"\n✅ Total rows: {len(df_saved)}")
print(f"\n📋 Column names:")
print(df_saved.columns.tolist())
print(f"\n🔍 First product sample:")
print(df_saved.iloc[0])
print(f"\n📝 Data types:")
print(df_saved.dtypes)





📊 Checking saved DataFrame...

✅ Total rows: 4566

📋 Column names:
['S.No', 'BrandName', 'Product ID', 'Product Name', 'Brand Desc', 'Product Size', 'Currancy', 'MRP', 'SellPrice', 'Discount', 'Category']

🔍 First product sample:
S.No                                    1
BrandName                            4711
Product ID                          FR001
Product Name            Cologne Fragrance
Brand Desc      ekw eau de cologne 400 ml
Product Size                        Small
Currancy                              Rs.
MRP                                  3900
SellPrice                            3120
Discount                          20% off
Category                  Fragrance-Women
Name: 0, dtype: object

📝 Data types:
S.No             int64
BrandName       object
Product ID      object
Product Name    object
Brand Desc      object
Product Size    object
Currancy        object
MRP             object
SellPrice        int64
Discount        object
Category        object
dtype: object


In [ ]:
# Cell 7: Manual Search Function (Interactive Testing)
import pickle
import numpy as np

# Load the saved data
print("📦 Loading FAISS index and model...")
with open('faiss_index.pkl', 'rb') as f:
    data = pickle.load(f)
    index = data['index']
    df = data['df']
    model = data['model']

print("✅ Loaded successfully!")
print(f"📊 Total products: {len(df)}")
print(f"🔍 Ready to search!\n")

# Define search function
def search_products(query, k=5):
    """
    Search for products using natural language query

    Args:
        query (str): Your search query
        k (int): Number of results to return (default: 5)

    Returns:
        Results with product details and similarity scores
    """
    print(f"🔎 Query: '{query}'")
    print("="*60)

    # Encode query to vector
    query_vec = model.encode([query])

    # Search in FAISS index
    distances, indices = index.search(np.array(query_vec), k)

    # Display results
    for i, idx in enumerate(indices[0], 1):
        product = df.iloc[idx]
        distance = distances[0][i-1]
        similarity = 100 - (distance * 10)
        similarity = max(0, min(100, similarity))

        print(f"\n{i}. {product.get('Product Name', 'N/A')}")
        print(f"   📝 Description: {product.get('Brand Desc', 'N/A')}")
        print(f"   🏷️  Category: {product.get('Category', 'N/A')}")
        print(f"   📏 Size: {product.get('Product Size', 'N/A')}")
        print(f"   💰 Price: {product.get('Currancy', 'Rs.')} {product.get('SellPrice', 'N/A')}")
        print(f"   🎉 Discount: {product.get('Discount', 'N/A')}")
        print(f"   📊 Match Score: {similarity:.1f}% (Distance: {distance:.4f})")

    print("\n" + "="*60)
    return indices[0], distances[0]

print("✅ Search function ready! Use it like this:")
print("   search_products('black dress for women', k=5)")

📦 Loading FAISS index and model...
✅ Loaded successfully!
📊 Total products: 4566
🔍 Ready to search!

✅ Search function ready! Use it like this:
   search_products('black dress for women', k=5)


In [ ]:
# Cell 9: Advanced Manual Search (Exactly Like Your Image)
import numpy as np

# Your query
query = "black dress for women"  # ← Change this!

# Encode query
query_vec = model.encode([query])

# Search (k=5 means top 5 results)
k = 5
distance, indices = index.search(np.array(query_vec), k)

print(f"Query: {query}\n")

# Display results
for i, idx in enumerate(indices[0]):
    product_name = df.iloc[idx].get('Product Name', 'N/A')
    product_desc = df.iloc[idx].get('Brand Desc', 'N/A')
    print(f"{i+1}. {product_name}")
    print(f"   {product_desc} (Distance={distance[0][i]:.4f})")
    print()

Query: black dress for women

1. AND672 - Westernwear-Women
   floral square neck polyester women flared gown - black (Distance=0.8765)

2. AND221 - Westernwear-Women
   solid polyester v neck womens flared dress - black (Distance=0.8966)

3. AND278 - Westernwear-Women
   striped v-neck cotton blend womens straight dress - black mix (Distance=0.9003)

4. AND496 - Westernwear-Women
   solid round neck polyester women flared gown - black (Distance=0.9125)

5. COVER STORY14 - Westernwear-Women
   printed v neck cotton womens dresses - black (Distance=0.9138)

